# 🚀 Comet Boundary: Kubernetes Demo Tour

Welcome to the interactive Kubernetes demo for Comet Boundary.
This notebook will guide you step-by-step through deploying the local application into a local Kubernetes cluster (`minikube`).

## Phase 1: Environment Validation

First, we ensure the base environment is running by checking for the `.env` file generated by `make replay`. This file contains the dynamic IDs configured in Boundary.

In [ ]:
import os

if not os.path.exists(".env"):
    print("❌ Error: .env file not found. Please run 'make replay' in your terminal first.")
else:
    # Load variables into environment
    with open(".env") as f:
        for line in f:
            if '=' in line:
                k, v = line.strip().split('=', 1)
                os.environ[k] = v
    print("✅ Base environment configuration loaded.")

## Phase 2: Cluster Initialization

We are spinning up a local Kubernetes cluster (`minikube`) to simulate a production frontend environment.

In [ ]:
%%bash
if ! minikube status >/dev/null 2>&1; then
    echo "Starting Minikube..."
    minikube start
else
    echo "✅ Minikube is already running."
fi

## Phase 3: Image Bridging

Minikube operates in its own isolated Docker environment. The `comet-boundary-backend:latest` image we are loading was built locally during the `make replay` initialization step (it is also published as a release on our [GitHub Repository](https://github.com/arsenyspb/comet-boundary)).

Since we are running this demo locally without an external container registry, we must explicitly load this image into Minikube's internal registry so the cluster can use it.

**NOTE:** This operation takes a few moments to transfer the image.

In [ ]:
%%bash
echo "Ensuring no running deployment is using the image in Minikube..."
kubectl delete deployment comet-boundary-comet-boundary --ignore-not-found 2>/dev/null || true
echo "Loading comet-boundary-backend:latest into Minikube (please wait)..."
minikube image load comet-boundary-backend:latest
echo "✅ Image successfully loaded into Minikube."

## Phase 4: Configuration & Deployment

In the [production environment](https://github.com/arsenyspb/comet-boundary/tree/main/terraform), HashiCorp Terraform is used to provision Boundary resources, host catalogs, and RBAC rules. 

Here is a comparison of the configuration workflows:

**Production Architecture:**
```text
[ Terraform Cloud / CI ] ---- Provisions ----> [ HashiCorp Boundary (HCP) ]
                                                         ^
                                                         |
[ EKS/GKE Cluster ] <------ Configures ------ [ Application Pods ]
```

**Minikube Demo Architecture:**
```text
[ setup-boundary.sh ] ---- Provisions ----> [ Local Boundary (Docker Compose) ]
                                                         ^
                                                         |
[ Minikube (Local) ] <------ Bridges ------ [ Application Pods ]
```

We are now going to deploy the Helm chart. We will inject the dynamic Auth Method IDs and bridge the Minikube network to communicate with the host's Boundary Controller.

In [ ]:
import os
import subprocess

print("Resolving Minikube host bridge IP for Boundary Worker...")
raw_output = subprocess.check_output("minikube ssh \"grep host.minikube.internal /etc/hosts\"", shell=True, text=True)
worker_ip = raw_output.split()[0].strip()
print(f"✅ Host IP resolved to: {worker_ip}")

cmd = f"""
helm upgrade --install comet-boundary ./charts/comet-boundary \
  --set image.repository=comet-boundary-backend \
  --set image.tag=latest \
  --set boundary.address="http://host.minikube.internal:9200" \
  --set boundary.authMethodId="{os.environ.get('BOUNDARY_AUTH_METHOD_ID')}" \
  --set boundary.ldapAuthMethodId="{os.environ.get('BOUNDARY_LDAP_AUTH_METHOD_ID')}" \
  --set boundary.workerIp="{worker_ip}" \
  --set service.type=ClusterIP \
  --set service.port=8080
"""

print("\nDeploying via Helm...")
subprocess.run(cmd, shell=True, check=True)

## Phase 5: Rollout & Access

We will wait for the Kubernetes deployment to become available. 
Once ready, we will print the credentials needed for the demo.

In [ ]:
import subprocess
import os

print("Waiting for the Kubernetes deployment to become available...")
subprocess.run("kubectl wait --for=condition=available deployment/comet-boundary-comet-boundary --timeout=60s", shell=True, check=True)

print("\n✅ Deployment Successful!")
print("\n🎉 Demo is Ready! 🎉")
print("======================================================")
print("🌐 Web App URL:       http://localhost:8080")
print("🛡️  Boundary Admin UI: http://localhost:9200")
print("------------------------------------------------------")
print(f"🔑 Admin Credentials:   admin / {os.environ.get('BOUNDARY_ADMIN_PASSWORD')}")
print("👥 LDAP Users (Team A): alice / changeme")
print("👥 LDAP Users (Team B): bob   / changeme")
print("👥 LDAP Users (Cross-Team): chris / changeme")
print("======================================================")

### Start the UI

To access the Web UI, run the cell below. It will start a background process to forward port 8080 from the Kubernetes cluster to your local machine. 

*When you are done testing, you can stop the cell execution to close the tunnel.*

In [ ]:
import subprocess
import webbrowser
import time

print("Starting port-forward on localhost:8080...")
process = subprocess.Popen(["kubectl", "port-forward", "svc/comet-boundary-comet-boundary", "8080:8080"])

time.sleep(2)
print("Opening browser...")
webbrowser.open("http://localhost:8080")

try:
    print("\n⏳ Port-forward is active. Interrupt this cell (Stop button) to close the connection.")
    process.wait()
except KeyboardInterrupt:
    print("\nStopping port-forward...")
    process.terminate()
    print("Connection closed.")